# 01A — Telecom sector pack

**Outcome:** translate the native synthetic Telecom files into the small
Pack v0.6 metric-level interface consumed by the common adapter.

This notebook owns Telecom meaning: native field names, metric units,
ONT identity and fault-label routing. It does not create model features.
`PACK-CORE` and `PACK-EVAL` are written to physically separate folders.
Each ONT stream receives an explicit `episode_id`. Network topology is
kept outside `PACK-CORE` and used only for split design, grouped-fault
evaluation and later incident localisation.


## 1. Setup

Keep this notebook and `milestone1_core.py` together in
`MyDrive/anomaly_detection/research/milestone1/`. The default run uses the
complete observable panel. Output directories are immutable, so change
`PACK_RUN_ID` before rebuilding a completed run.


In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "milestone1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    EVAL_SCHEMAS,
    PACK_ENTITY_SCHEMA,
    PACK_EPISODE_SCHEMA,
    PACK_METRIC_SCHEMA,
    PACK_OBSERVATION_SCHEMA,
    SPLIT_SCHEMAS,
    save_pack,
    new_output_directory,
    pack_fingerprint,
    read_json,
    source_file,
)

SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    DRIVE_ROOT / "telco_syntetic_data",
))
SOURCE_INSTANCE_ID = os.getenv(
    "TELECOM_SOURCE_INSTANCE_ID", "telemetry_synth_4_1_0_run_1"
)
PACK_RUN_ID = os.getenv("TELECOM_PACK_RUN_ID", "telecom_pack_v0_6_2")
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / "telecom" / PACK_RUN_ID
BATCH_ROWS = int(os.getenv("TELECOM_BATCH_ROWS", "100000"))
ENTITY_IDS = tuple(filter(None, os.getenv("TELECOM_ENTITY_IDS", "").split(",")))
SAMPLE_START = os.getenv("TELECOM_SAMPLE_START") or None
SAMPLE_END = os.getenv("TELECOM_SAMPLE_END") or None
RUN_BUILD = os.getenv("RUN_TELECOM_PACK", "1") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "source_instance_id": SOURCE_INSTANCE_ID,
    "pack_root": str(PACK_ROOT),
    "entities": ENTITY_IDS or "all",
    "sample_start": SAMPLE_START or "first observation",
    "sample_end": SAMPLE_END or "last observation",
    "batch_rows": BATCH_ROWS,
}, name="value").to_frame())


## 2. Telecom phrasebook

`native_field` is used only while reading the source. The remaining fields
form the authored metric catalogue. Every metric is periodic at 900 seconds.

The sector pack assigns simple source-quality codes: null values are invalid,
and FEC values at the generator ceiling are clipped.


In [ ]:
METRIC_COLUMNS = ["native_field", *PACK_METRIC_SCHEMA]
metric_map = pd.DataFrame([
    ("rx_power_dbm", "rx_power_dbm", "ont", "gauge", "dBm", "periodic", 900),
    ("tx_power_dbm", "tx_power_dbm", "ont", "gauge", "dBm", "periodic", 900),
    ("temperature_c", "temperature_c", "ont", "gauge", "degC", "periodic", 900),
    ("bias_current_ma", "bias_current_ma", "ont", "gauge", "mA", "periodic", 900),
    ("voltage_v", "voltage_v", "ont", "gauge", "V", "periodic", 900),
    ("ber", "ber", "ont", "bounded_fraction", "ratio", "periodic", 900),
    ("fec_count", "fec_count", "ont", "interval_count", "count", "periodic", 900),
    ("crc_errors", "crc_errors", "ont", "interval_count", "count", "periodic", 900),
    ("uptime_s", "uptime_s", "ont", "cumulative_counter", "s", "periodic", 900),
    ("reboot_count", "reboot_count", "ont", "cumulative_counter", "count", "periodic", 900),
    ("throughput_mbps", "throughput_mbps", "ont", "gauge", "Mbps", "periodic", 900),
], columns=METRIC_COLUMNS)

TOPOLOGY_GROUPS = [
    "olt_id", "pon_port", "splitter_l1",
    "splitter_l2", "geo_cluster",
]

assert metric_map["metric_id"].is_unique
display(metric_map)


## 3. Inspect the native source

Only the timestamp, ONT identifier and explicitly mapped measurements are
read into `PACK-CORE`. Unmapped source columns are displayed for review but
are not maintained in a hard-coded allowlist. The pack fails if the
observable panel still contains recognisable truth fields.

The fault registry and fault intervals are evaluation-only. Tickets are
deferred until operator-oriented evaluation. The five approved topology
fields are required for this Telecom fixture and are written only to `SPLITS`.


In [ ]:
def iter_panel(path, columns, batch_rows):
    parquet = pq.ParquetFile(path)
    for batch in parquet.iter_batches(batch_size=batch_rows, columns=columns):
        yield batch.to_pandas()


PANEL_PATH = SOURCE / "reference_dataset.parquet"
FAULT_REGISTRY_PATH = SOURCE / "gt_fault_registry.csv"
FAULT_INTERVALS_PATH = SOURCE / "fault_entity_intervals.csv"
TOPOLOGY_PATH = SOURCE / "topology.csv"

if not PANEL_PATH.is_file():
    raise FileNotFoundError(f"Missing Telecom telemetry: {PANEL_PATH}")

native_columns = pq.ParquetFile(PANEL_PATH).schema_arrow.names
truth_tokens = ("fault", "label", "anomaly", "root_cause", "ticket")
truth_columns = sorted(
    name for name in native_columns
    if name.lower().startswith("gt_")
    or name.lower() in {"class", "state"}
    or any(token in name.lower() for token in truth_tokens)
)
available_metrics = metric_map.loc[
    metric_map["native_field"].isin(native_columns)
].copy()
missing_metrics = sorted(
    set(metric_map["native_field"]) - set(native_columns)
)

unmapped_columns = sorted(
    set(native_columns)
    - {"timestamp_utc", "ont_id"}
    - set(available_metrics["native_field"])
)

fault_registry_exists = FAULT_REGISTRY_PATH.is_file()
fault_intervals_exist = FAULT_INTERVALS_PATH.is_file()
if fault_registry_exists != fault_intervals_exist:
    raise FileNotFoundError(
        "Evaluation requires both the fault registry and fault intervals."
    )
EVALUATION_READY = fault_registry_exists and fault_intervals_exist

inventory = {
    "panel": PANEL_PATH.name,
    "panel_columns": len(native_columns),
    "mapped_metrics": len(available_metrics),
    "missing_expected_metrics": missing_metrics,
    "unmapped_columns_not_used_as_telemetry": unmapped_columns,
    "truth_columns_in_observable_panel": truth_columns,
    "evaluation_ready": EVALUATION_READY,
    "topology_available": TOPOLOGY_PATH.is_file(),
}
display(pd.Series(inventory, name="value").to_frame())

missing_columns = {"timestamp_utc", "ont_id"} - set(native_columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
if truth_columns:
    raise ValueError(f"Truth fields found in observable telemetry: {truth_columns}")
if missing_metrics:
    raise ValueError(f"Expected Telecom metrics are missing: {missing_metrics}")
if not TOPOLOGY_PATH.is_file():
    raise FileNotFoundError(f"Missing Telecom topology: {TOPOLOGY_PATH}")
topology_columns = pd.read_csv(TOPOLOGY_PATH, nrows=0).columns
missing_topology = {"ont_id", *TOPOLOGY_GROUPS} - set(topology_columns)
if missing_topology:
    raise ValueError(f"Missing topology columns: {sorted(missing_topology)}")


## 4. Build the Telecom pack

The pack writes one row per metric observation. This makes observation
presence explicit and permits future sectors to use different cadences by
metric. The entity registry identifies monitored
ONTs only; its observation bounds are derived later by the common adapter.


In [ ]:
def select_rows(frame, entity_ids=(), start=None, end=None):
    selected = frame.copy()
    if entity_ids:
        selected = selected.loc[selected["ont_id"].astype(str).isin(entity_ids)]
    timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if start:
        selected = selected.loc[timestamps.ge(pd.to_datetime(start, utc=True))]
        timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if end:
        selected = selected.loc[timestamps.lt(pd.to_datetime(end, utc=True))]
    return selected.reset_index(drop=True)


def telecom_time_partitions(timestamps):
    timestamps = pd.Series(pd.to_datetime(sorted(set(timestamps)), utc=True))
    if len(timestamps) < 4:
        raise ValueError("At least four timestamps are required for temporal splits")
    boundaries = [
        0,
        round(len(timestamps) * 0.50),
        round(len(timestamps) * 0.75),
        len(timestamps),
    ]
    names = ["calibration", "development", "holdout"]
    rows = []
    for index, name in enumerate(names):
        start_index, end_index = boundaries[index:index + 2]
        end_ts = (
            timestamps.iloc[end_index]
            if end_index < len(timestamps)
            else timestamps.iloc[-1] + pd.Timedelta(seconds=900)
        )
        rows.append((name, timestamps.iloc[start_index], end_ts, "telecom_time_v1"))
    return pd.DataFrame(rows, columns=SPLIT_SCHEMAS["time_partitions"])


def telecom_entity_groups(topology, entity_ids):
    entity_ids = set(map(str, entity_ids))
    selected = topology.loc[
        topology["ont_id"].astype(str).isin(entity_ids)
    ].copy()
    selected["ont_id"] = selected["ont_id"].astype(str)
    missing_entities = entity_ids - set(selected["ont_id"])
    if missing_entities:
        raise ValueError(
            f"Topology is missing {len(missing_entities)} selected ONTs"
        )
    missing_memberships = selected[["ont_id", *TOPOLOGY_GROUPS]].isna()
    if missing_memberships.any().any():
        raise ValueError(
            "Selected ONTs have incomplete topology memberships"
        )
    rows = []
    for group_type in TOPOLOGY_GROUPS:
        pairs = selected[["ont_id", group_type]].dropna().drop_duplicates()
        if pairs.groupby("ont_id")[group_type].nunique().gt(1).any():
            raise ValueError(f"An ONT maps to multiple {group_type} groups")
        rows.extend(
            (str(entity_id), group_type, str(group_id), "telecom_topology_v1")
            for entity_id, group_id in pairs.itertuples(index=False)
        )
    return pd.DataFrame(rows, columns=SPLIT_SCHEMAS["entity_groups"])


def telecom_evaluation(root, selected_entities):
    selected_entities = set(map(str, selected_entities))
    registry = pd.read_csv(Path(root) / "gt_fault_registry.csv")
    intervals = pd.read_csv(Path(root) / "fault_entity_intervals.csv")
    required_registry = {
        "gt_fault_id", "gt_fault_type", "target", "onset_ts",
        "first_observable_ts", "impact_ts", "repair_ts", "group_id",
    }
    required_intervals = {
        "fault_id", "entity_id", "active_start_ts", "active_end_ts",
    }
    missing_registry = required_registry - set(registry.columns)
    missing_intervals = required_intervals - set(intervals.columns)
    if missing_registry:
        raise ValueError(f"Missing fault-registry columns: {sorted(missing_registry)}")
    if missing_intervals:
        raise ValueError(f"Missing fault-interval columns: {sorted(missing_intervals)}")

    intervals = intervals.loc[
        intervals["entity_id"].astype(str).isin(selected_entities)
    ].copy()
    selected_faults = set(intervals["fault_id"].dropna().astype(str))
    registered_faults = set(registry["gt_fault_id"].dropna().astype(str))
    missing_faults = selected_faults - registered_faults
    if missing_faults:
        raise ValueError(
            "Fault intervals reference missing registry faults: "
            f"{sorted(missing_faults)[:10]}"
        )
    registry = registry.loc[
        registry["gt_fault_id"].astype(str).isin(selected_faults)
    ].copy()

    fault_events = pd.DataFrame({
        "fault_id": registry["gt_fault_id"].astype("string"),
        "fault_type": registry["gt_fault_type"].astype("string"),
        "domain_id": registry["target"].astype("string"),
        "onset_ts": registry["onset_ts"],
        "observable_ts": registry["first_observable_ts"],
        "impact_ts": registry["impact_ts"],
        "end_ts": registry["repair_ts"],
        "group_id": registry["group_id"].astype("string"),
        "label_source": "synthetic_generator_truth",
        "source_instance_id": pd.NA,
    })
    fault_intervals = pd.DataFrame({
        "fault_id": intervals["fault_id"].astype("string"),
        "entity_id": intervals["entity_id"].astype("string"),
        "start_ts": intervals["active_start_ts"],
        "end_ts": intervals["active_end_ts"],
        "label_source": "synthetic_generator_truth",
        "source_instance_id": pd.NA,
    })
    tables = {
        "fault_events": fault_events,
        "fault_entity_intervals": fault_intervals,
    }
    for table_name, frame in tables.items():
        for column in frame:
            if column.endswith("_ts"):
                frame[column] = pd.to_datetime(
                    frame[column], utc=True, errors="raise"
                )
        tables[table_name] = frame[EVAL_SCHEMAS[table_name]]
    return tables


In [ ]:
def build_telecom_pack(
    source,
    destination,
    *,
    source_instance_id,
    include_evaluation=True,
    entity_ids=(),
    start=None,
    end=None,
    batch_rows=100_000,
):
    source, destination = Path(source), Path(destination)
    source_instance_id = str(source_instance_id).strip()
    if not source_instance_id:
        raise ValueError("source_instance_id must not be empty")
    episode_prefix = f"{source_instance_id}::"
    panel = source / "reference_dataset.parquet"
    if not panel.is_file():
        raise FileNotFoundError(f"Missing Telecom telemetry: {panel}")
    panel_fields = pq.ParquetFile(panel).schema_arrow.names
    missing_metrics = sorted(
        set(metric_map["native_field"]) - set(panel_fields)
    )
    if missing_metrics:
        raise ValueError(
            f"Expected Telecom metrics are missing: {missing_metrics}"
        )
    catalogue = metric_map.copy()
    read_columns = ["timestamp_utc", "ont_id", *catalogue["native_field"]]
    rename_metrics = dict(zip(catalogue["native_field"], catalogue["metric_id"]))
    registry_path = source / "gt_fault_registry.csv"
    intervals_path = source / "fault_entity_intervals.csv"
    if registry_path.is_file() != intervals_path.is_file():
        raise FileNotFoundError(
            "Evaluation requires both the fault registry and intervals, or neither."
        )
    evaluation_available = registry_path.is_file() and intervals_path.is_file()
    if include_evaluation and not evaluation_available:
        raise FileNotFoundError(
            "Evaluation was requested, but the required Telecom truth "
            "files are missing."
        )

    with new_output_directory(destination) as pack:
        core = pack / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        observed_entities = set()
        observed_timestamps = set()
        part_number = 0
        for batch in iter_panel(panel, read_columns, batch_rows):
            batch = select_rows(batch, entity_ids, start, end)
            if batch.empty:
                continue
            wide = batch.rename(columns={
                "timestamp_utc": "event_ts",
                "ont_id": "entity_id",
                **rename_metrics,
            })
            wide["event_ts"] = pd.to_datetime(wide["event_ts"], utc=True)
            wide["entity_id"] = wide["entity_id"].astype(str)
            wide.insert(2, "episode_id", episode_prefix + wide["entity_id"])
            wide = wide[["event_ts", "entity_id", "episode_id", *catalogue["metric_id"]]]
            long = wide.melt(
                id_vars=["event_ts", "entity_id", "episode_id"],
                value_vars=catalogue["metric_id"],
                var_name="metric_id",
                value_name="value",
            )
            long["quality_code"] = "measured"
            long.loc[long["value"].isna(), "quality_code"] = "invalid"
            fec_clipped = long["metric_id"].eq("fec_count") & long["value"].ge(5_000_000)
            long.loc[fec_clipped, "quality_code"] = "clipped"
            long = long[PACK_OBSERVATION_SCHEMA]
            long.to_parquet(
                observations / f"part-{part_number:05d}.parquet",
                index=False,
                compression="zstd",
            )
            observed_entities.update(long["entity_id"].unique())
            observed_timestamps.update(long["event_ts"].unique())
            part_number += 1

        if not observed_entities:
            raise ValueError("The selected telemetry slice is empty")

        catalogue[PACK_METRIC_SCHEMA].to_parquet(
            core / "metric_catalogue.parquet", index=False
        )
        pd.DataFrame({
            "entity_id": sorted(observed_entities),
            "entity_type": "ont",
        })[PACK_ENTITY_SCHEMA].to_parquet(core / "entity_registry.parquet", index=False)
        pd.DataFrame({
            "episode_id": [
                episode_prefix + entity_id
                for entity_id in sorted(observed_entities)
            ],
            "entity_id": sorted(observed_entities),
            "episode_basis": "single_generator_run",
        })[PACK_EPISODE_SCHEMA].to_parquet(
            core / "observation_episodes.parquet", index=False
        )

        splits = pack / "SPLITS"
        splits.mkdir()
        telecom_time_partitions(observed_timestamps).to_parquet(
            splits / "time_partitions.parquet", index=False
        )
        split_tables = ["time_partitions", "entity_groups"]
        topology_path = source / "topology.csv"
        if not topology_path.is_file():
            raise FileNotFoundError(f"Missing Telecom topology: {topology_path}")
        topology = pd.read_csv(
            topology_path, usecols=["ont_id", *TOPOLOGY_GROUPS]
        )
        entity_groups = telecom_entity_groups(topology, observed_entities)
        entity_groups.to_parquet(splits / "entity_groups.parquet", index=False)

        evaluation_tables = []
        if include_evaluation:
            evaluation = pack / "PACK-EVAL"
            evaluation.mkdir()
            tables = telecom_evaluation(source, observed_entities)
            evaluation_tables = list(tables)
            for name, frame in tables.items():
                frame.to_parquet(evaluation / f"{name}.parquet", index=False)

        source_files = [source_file(panel, source, "model_input")]
        source_files.append(source_file(
            topology_path, source, "incident_grouping_and_evaluation"
        ))
        if include_evaluation:
            source_files.extend([
                source_file(registry_path, source, "evaluation_only"),
                source_file(intervals_path, source, "evaluation_only"),
            ])

        save_pack(
            pack,
            sector="telecom",
            pack_version="0.6.2",
            source_info={
                "source_id": "telemetry-synth-4.1.0",
                "source_instance_id": source_instance_id,
                "source_root": str(source),
                "files": source_files,
                "selection": {
                    "entity_ids": list(entity_ids),
                    "start": start,
                    "end": end,
                },
            },
            evaluation_tables=evaluation_tables,
            split_tables=split_tables,
            notes=[
                "Null values are marked invalid.",
                "FEC values at 5000000 are marked clipped.",
                "Tickets are deferred until operator-oriented evaluation.",
                "One source-run episode is declared per ONT.",
                "Episode boundaries are not inferred from telemetry gaps.",
                "Observation presence is explicit at metric level.",
                "OLT, PON, splitter and geographic groups are stored in SPLITS.",
                "Topology is excluded from PACK-CORE and detector inputs.",
            ],
        )
    return read_json(destination / "pack_manifest.json")


if RUN_BUILD:
    pack_manifest = build_telecom_pack(
        SOURCE,
        PACK_ROOT,
        source_instance_id=SOURCE_INSTANCE_ID,
        include_evaluation=True,
        entity_ids=ENTITY_IDS,
        start=SAMPLE_START,
        end=SAMPLE_END,
        batch_rows=BATCH_ROWS,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(pack_manifest["core_row_counts"], name="rows").to_frame())


## 5. Truth-isolation test

This is the primary leakage test because the sector notebook is the only
component that sees native observations and native truth together.

The same observable fixture is translated twice: once with native evaluation
files and the original topology, and once after evaluation files and topology
truth columns are removed. `PACK-CORE` fingerprints and approved topology
groups must be identical. A deliberately leaky signature must change.


In [ ]:
def write_small_panel(source_panel, destination, rows=5_000):
    columns = ["timestamp_utc", "ont_id", *available_metrics["native_field"]]
    sample = next(iter_panel(source_panel, columns, rows)).head(rows)
    sample.to_parquet(destination / "reference_dataset.parquet", index=False)
    return sample


def make_isolation_fixture(source, destination, include_evaluation):
    destination.mkdir()
    write_small_panel(Path(source) / "reference_dataset.parquet", destination)
    topology = pd.read_csv(Path(source) / "topology.csv")
    if not include_evaluation:
        topology = topology[["ont_id", *TOPOLOGY_GROUPS]]
    topology.to_csv(destination / "topology.csv", index=False)
    if include_evaluation:
        for name in ("gt_fault_registry.csv", "fault_entity_intervals.csv", "tickets.csv"):
            path = Path(source) / name
            if path.is_file():
                shutil.copy2(path, destination / name)


def deliberately_leaky_signature(source):
    topology_path = Path(source) / "topology.csv"
    if topology_path.is_file():
        truth_fields = sorted(
            column for column in pd.read_csv(topology_path, nrows=0).columns
            if column.lower().startswith("gt_")
        )
        if truth_fields:
            return tuple(truth_fields)
    for name in ("tickets.csv", "gt_fault_registry.csv"):
        path = Path(source) / name
        if path.is_file():
            return len(pd.read_csv(path))
    return "missing"


def run_truth_isolation_test():
    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        original_source = temporary / "native_original"
        redacted_source = temporary / "native_redacted"
        make_isolation_fixture(SOURCE, original_source, include_evaluation=True)
        make_isolation_fixture(SOURCE, redacted_source, include_evaluation=False)

        original_pack = temporary / "pack_original"
        redacted_pack = temporary / "pack_redacted"
        build_telecom_pack(
            original_source, original_pack,
            source_instance_id=SOURCE_INSTANCE_ID,
            include_evaluation=True,
        )
        build_telecom_pack(
            redacted_source, redacted_pack,
            source_instance_id=SOURCE_INSTANCE_ID,
            include_evaluation=False,
        )
        try:
            build_telecom_pack(
                redacted_source, temporary / "pack_truth_required",
                source_instance_id=SOURCE_INSTANCE_ID,
                include_evaluation=True,
            )
        except FileNotFoundError:
            pass
        else:
            raise AssertionError(
                "Requested evaluation did not fail when truth was absent"
            )

        assert pack_fingerprint(original_pack) == pack_fingerprint(redacted_pack)
        original_groups = pd.read_parquet(
            original_pack / "SPLITS" / "entity_groups.parquet"
        ).sort_values(SPLIT_SCHEMAS["entity_groups"]).reset_index(drop=True)
        redacted_groups = pd.read_parquet(
            redacted_pack / "SPLITS" / "entity_groups.parquet"
        ).sort_values(SPLIT_SCHEMAS["entity_groups"]).reset_index(drop=True)
        pd.testing.assert_frame_equal(original_groups, redacted_groups)
        assert deliberately_leaky_signature(original_source) != deliberately_leaky_signature(redacted_source)

if EVALUATION_READY:
    run_truth_isolation_test()
    print("PASS — PACK-CORE is unchanged after evaluation files are removed")
    print("PASS — topology groups are unchanged after topology truth is removed")
    print("PASS — the negative control detects removed evaluation truth")
    print("PASS — requested evaluation fails when truth files are absent")
else:
    print("NOT RUN — truth isolation requires a labelled development fixture")


## 6. Inspect the result

The tables below are small previews. Topology groups are shown separately
because they are not detector inputs. The manifest contains one fingerprint,
source roles and row counts.


In [ ]:
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "metric_catalogue.parquet"))
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "entity_registry.parquet").head())
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "observation_episodes.parquet").head())
display(pd.read_parquet(PACK_ROOT / "SPLITS" / "time_partitions.parquet"))
entity_groups = pd.read_parquet(PACK_ROOT / "SPLITS" / "entity_groups.parquet")
display(entity_groups.groupby("group_type")["group_id"].nunique().rename("groups").to_frame())
display(entity_groups.head(15))
display(pd.Series(read_json(PACK_ROOT / "pack_manifest.json"), name="value").to_frame())
print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")
